# 10 — Random Forest: 8-Fold Signer-Independent Training and Evaluation

**Goal.** Train and evaluate the engineered-feature baseline on both branches under the frozen folds. It runs 16 primary scopes (8 folds × 2 branches) plus 2 signer-overlap controls. There is no inferential statistics here; that is in 13.

**Frozen model**
```
RandomForestClassifier(n_estimators=300, criterion="gini", max_depth=None,
                       min_samples_split=2, min_samples_leaf=1, max_features=96,
                       bootstrap=True, class_weight=None, n_jobs=-1, random_state=seed)
```
Seeds: fold *k* → `42000 + k`, control → `42999`, the same for both branches within a fold.

**Key design decisions**
- **Train on the 12 training signers only.** RF needs no early stopping, but the neural models use the 2 validation signers for training control, so RF uses the same 12 to stay comparable. Validation predictions are saved as diagnostics only. There is no retraining on train + validation.
- **`max_features = 96` is held equal across branches.** This is not neutral: it is ≈1.0× sqrt(D) for Reduced but ≈0.61× for Full, so it is potentially **conservative toward Full**. `"sqrt"` is a pre-declared secondary sensitivity analysis and was not run.
- **Resource gate.** With 200 classes, forest memory (not time) is the constraint. 25-tree probes on fold 0 project the 300-tree size, and the notebook stops before any evaluation if it exceeds 8 GiB.
- No scaling, PCA or selection beyond the 09 zero-variance filter.
- Resumable: each scope writes its artefact atomically and is reused only if its contract, provenance and prediction digest match.

**Inputs → Outputs.** 09, 08, 07 → `rf_oof_{b}.npz`, `rf_control_{b}.npz`, `rf_fold_metrics.csv`, `rf_pooled_metrics.csv`, `rf_control_metrics.csv`, `rf_efficiency.csv`, `rf_class_recall.csv`, confusion matrices, feature importances, `rf_model_contract.json`.

In [ ]:
# ============================================================
# CELL 1 — Configuration and the frozen RF policy
# ============================================================
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import resource
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, recall_score)

NOTEBOOK = "nb10"
RF_POLICY_VERSION = "nb10-v1"
OVERWRITE = False

# ---- frozen upstream provenance --------------------------------------------
REQUIRED_EXPERIMENT_ID = "1308155924042fc2"
REQUIRED_LOADER_CONTRACT_ID = "112c4e5a438540d0"
REQUIRED_FEATURE_CONTRACT_ID = "bfcce8a5fa013fab"
REQUIRED_FILTER_SET_ID = "3b5f7a8bfc08656c"
REQUIRED_TIME_METADATA_ID = "e30a51110d2f9324"
REQUIRED_FEATURE_MODULE_SHA256 = (
    "c7170a40a75a44602a0e9ba26f3622396a6d8cc88d92a84acc5e26421ba2117d")
REQUIRED_LOADER_MODULE_SHA256 = (
    "3ec9bf132b3ada3a3d2fd2379b14e112b5d2a3761adad04af75133f168e7f55b")
REQUIRED_SPEC_HASH = {"full": "092fac3aa5e587f5", "reduced": "ba1a9b43de9f8301"}
REQUIRED_MATRIX_DIGEST = {"full": "14374f55f7324252", "reduced": "39bcc7e46bce35e9"}

BRANCHES: Tuple[str, ...] = ("full", "reduced")
N_FOLDS, N_CLIPS, N_LABELS, N_SIGNERS = 8, 6000, 200, 16
TRAIN_SIGNERS_PER_FOLD = 12
CONTROL_SPLIT_SIZES = {"train": 4500, "validation": 750, "test": 750}
FEATURE_COUNT = {"full": 25168, "reduced": 9000}

# Retained dimensions come from the frozen Notebook 9 filters and are asserted,
# never recomputed.
EXPECTED_RETAINED = {
    "full": {0: 25079, 1: 25079, 2: 25082, 3: 25082, 4: 25082, 5: 25082,
             6: 25082, 7: 25082, "control": 25082},
    "reduced": {0: 8911, 1: 8911, 2: 8914, 3: 8914, 4: 8914, 5: 8914,
                6: 8914, 7: 8914, "control": 8914},
}

# ---- the frozen RF policy. Fixed before any result is seen. -----------------
MAX_FEATURES_PRIMARY = 96
N_ESTIMATORS = 300
RF_PARAMS: Dict[str, Any] = {
    "n_estimators": N_ESTIMATORS, "criterion": "gini", "max_depth": None,
    "min_samples_split": 2, "min_samples_leaf": 1,
    "max_features": MAX_FEATURES_PRIMARY, "bootstrap": True,
    "class_weight": None, "n_jobs": -1,
}
BENCHMARK_ESTIMATORS = 25
# Compute-safety gate, frozen before training. With 200 classes each tree stores a
# (n_nodes, 1, 200) float64 value array, so a 300-tree forest can be several GiB.
# The 25-tree probe projects that, and exceeding this ceiling STOPS the notebook
# before any validation or test prediction exists. n_estimators is never changed
# automatically.
MAX_PROJECTED_FOREST_VALUE_GIB = 8.0
FOLD_SEED_BASE = 42000
CONTROL_SEED = 42999
FPS_REGIME_BOUNDARY = 40.0          # the Notebook 9 definition: < 40 is ~30 fps
REGIME_LABELS = ("~30 fps", "~60 fps")
TOP_K = 5

SEARCH_ROOT = "/kaggle/input"
WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
SCOPE_DIR = WORKING / "rf_scopes"
SCOPE_DIR.mkdir(parents=True, exist_ok=True)

OUT = {
    "contract": WORKING / "rf_model_contract.json",
    "manifest": WORKING / "rf_model_manifest.json",
    "fold_metrics": WORKING / "rf_fold_metrics.csv",
    "validation_metrics": WORKING / "rf_validation_metrics.csv",
    "pooled_metrics": WORKING / "rf_pooled_metrics.csv",
    "signer_metrics": WORKING / "rf_signer_metrics.csv",
    "class_recall": WORKING / "rf_class_recall.csv",
    "control_metrics": WORKING / "rf_control_metrics.csv",
    "efficiency": WORKING / "rf_efficiency.csv",
    "importance_summary": WORKING / "rf_importance_summary.csv",
    "report_json": WORKING / "rf_validation_report.json",
    "report_csv": WORKING / "rf_validation_report.csv",
    "preflight_report": WORKING / "rf_preflight_report.json",
}
OOF_PATH = {b: WORKING / f"rf_oof_{b}.npz" for b in BRANCHES}
CONTROL_PATH = {b: WORKING / f"rf_control_{b}.npz" for b in BRANCHES}
CONFUSION_PATH = {b: WORKING / f"rf_confusion_matrix_{b}.npy" for b in BRANCHES}
IMPORTANCE_PATH = {b: WORKING / f"rf_feature_importances_{b}.npy" for b in BRANCHES}

ENVIRONMENT = {
    "notebook": NOTEBOOK, "rf_policy_version": RF_POLICY_VERSION,
    "python": platform.python_version(), "numpy": np.__version__,
    "pandas": pd.__version__, "sklearn": sklearn.__version__,
    "cpu_count": os.cpu_count(),
}

STARTED = time.time()
CHECKS: List[Dict[str, Any]] = []


def record(name: str, passed: bool, detail: Any = "", status: str = "") -> bool:
    passed = bool(passed)
    status = status or ("pass" if passed else "fail")
    CHECKS.append({"check": name, "passed": passed, "status": status,
                   "detail": str(detail)})
    label = {"pass": "PASS", "fail": "FAIL", "skip": "SKIP"}[status]
    print(f"  {label}  {name:<62} {detail}")
    return passed


def require(name: str, passed: bool, detail: Any = "") -> None:
    if not record(name, passed, detail):
        raise RuntimeError(f"{name} failed ({detail}). Notebook 10 refuses to train or "
                           "publish on top of a failed check.")


def skip(name: str, reason: Any) -> None:
    record(name, True, reason, status="skip")


def failures() -> List[Dict[str, Any]]:
    return [entry for entry in CHECKS if entry["status"] == "fail"]


def sha16(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]


def array_digest(*arrays: np.ndarray) -> str:
    digest = hashlib.sha256()
    for array in arrays:
        contiguous = np.ascontiguousarray(array)
        digest.update(str(contiguous.dtype).encode("utf-8"))
        digest.update(str(contiguous.shape).encode("utf-8"))
        digest.update(contiguous.tobytes())
    return digest.hexdigest()[:16]


def peak_memory_gib() -> float:
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024 ** 2)


def atomic_write(path: Path, writer) -> None:
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        writer(handle)
    os.replace(temporary, path)


def scope_seed(scope: Any) -> int:
    return CONTROL_SEED if scope == "control" else FOLD_SEED_BASE + int(scope)


def scope_label(branch: str, scope: Any) -> str:
    return f"{branch}_{'control' if scope == 'control' else f'fold{scope}'}"


print("Environment")
for key, value in ENVIRONMENT.items():
    print(f"  {key}: {value}")

print("\nRequired upstream provenance")
for name, value in (("experiment id", REQUIRED_EXPERIMENT_ID),
                    ("loader contract id", REQUIRED_LOADER_CONTRACT_ID),
                    ("feature contract id", REQUIRED_FEATURE_CONTRACT_ID),
                    ("filter set id", REQUIRED_FILTER_SET_ID),
                    ("time metadata id", REQUIRED_TIME_METADATA_ID)):
    print(f"  {name:<22}: {value}")
for branch in BRANCHES:
    print(f"  {branch:<8} spec hash {REQUIRED_SPEC_HASH[branch]}  "
          f"matrix digest {REQUIRED_MATRIX_DIGEST[branch]}")

print("\nFrozen RF policy (fixed before any result is seen)")
for key, value in RF_PARAMS.items():
    print(f"  {key:<20}: {value}")
print(f"  random_state        : fold k -> {FOLD_SEED_BASE} + k, control -> {CONTROL_SEED}")
print(f"  resource gate       : projected forest value arrays must stay within "
      f"{MAX_PROJECTED_FOREST_VALUE_GIB} GiB")
print(f"\n  MAX_FEATURES_PRIMARY = {MAX_FEATURES_PRIMARY}")
print(f"  reduced natural sqrt ~ {np.sqrt(EXPECTED_RETAINED['reduced'][2]):.0f}  ->  "
      f"{MAX_FEATURES_PRIMARY / np.sqrt(EXPECTED_RETAINED['reduced'][2]):.2f}x")
print(f"  full    natural sqrt ~ {np.sqrt(EXPECTED_RETAINED['full'][2]):.0f}  ->  "
      f"{MAX_FEATURES_PRIMARY / np.sqrt(EXPECTED_RETAINED['full'][2]):.2f}x")
print("  Equal absolute search breadth is NOT neutral: it is potentially conservative")
print("  toward Full in per-split search breadth relative to its natural sqrt setting.")
print("  max_features='sqrt' is pre-registered as a SECONDARY sensitivity analysis, is")
print("  not run here, and never replaces this primary result.")

print(f"\n  raw engineered dimension ratio     : "
      f"{FEATURE_COUNT['full'] / FEATURE_COUNT['reduced']:.3f}x "
      f"({FEATURE_COUNT['full']:,} / {FEATURE_COUNT['reduced']:,})")
print(f"  post-zero-variance model-input ratio: "
      f"{EXPECTED_RETAINED['full'][2] / EXPECTED_RETAINED['reduced'][2]:.3f}x "
      f"({EXPECTED_RETAINED['full'][2]:,} / {EXPECTED_RETAINED['reduced'][2]:,})")

print(f"\n  scopes: {N_FOLDS} folds x {len(BRANCHES)} branches + "
       f"{len(BRANCHES)} control = {N_FOLDS * len(BRANCHES) + len(BRANCHES)}")
print(f"  OVERWRITE = {OVERWRITE}  (completed scopes are resumed, never silently reused)")
print("\nTraining uses the 12 training signers only. Validation predictions are saved")
print("and labelled DIAGNOSTIC ONLY. No inferential statistics are computed here.")

In [ ]:
# ============================================================
# CELL 2 — Provenance: hash before import, verify every frozen id
# ============================================================

print("LOCATING FROZEN OUTPUTS")


def find_one(pattern: str, description: str, root: str = SEARCH_ROOT) -> Path:
    hits = sorted(Path(root).glob(pattern))
    if not hits:
        raise FileNotFoundError(f"No {description} found under {root} "
                                f"(pattern {pattern!r}). Attach the producing notebook "
                                "output.")
    if len(hits) > 1:
        listing = "\n".join(f"    {h}" for h in hits)
        raise RuntimeError(f"{len(hits)} candidates for {description}; refusing to "
                           f"choose.\n{listing}")
    return hits[0]


FEATURE_MANIFEST_PATH = find_one("**/rf_feature_manifest.json", "notebook 9 manifest")
FEATURE_MODULE_PATH = find_one("**/bsl_rf_features.py", "notebook 9 feature module")
LOADER_MANIFEST_PATH = find_one("**/loader_manifest.json", "notebook 8 manifest")
LOADER_MODULE_PATH = find_one("**/bsl_sequence_loader.py", "notebook 8 loader module")
NB9_DIR, NB8_DIR = FEATURE_MANIFEST_PATH.parent, LOADER_MANIFEST_PATH.parent
NB7_DIR = find_one("**/experiment_manifest.json", "notebook 7 experiment manifest").parent
print(f"  notebook 9: {NB9_DIR}")
print(f"  notebook 8: {NB8_DIR}")
print(f"  notebook 7: {NB7_DIR}")

# ---- hash both modules BEFORE importing either -----------------------------
FEATURE_MODULE_SHA256 = hashlib.sha256(FEATURE_MODULE_PATH.read_bytes()).hexdigest()
LOADER_MODULE_SHA256 = hashlib.sha256(LOADER_MODULE_PATH.read_bytes()).hexdigest()
FEATURE_MANIFEST = json.loads(FEATURE_MANIFEST_PATH.read_text(encoding="utf-8"))
LOADER_MANIFEST = json.loads(LOADER_MANIFEST_PATH.read_text(encoding="utf-8"))

print("\nMODULE INTEGRITY (checked before import)")
require("feature module matches the frozen sha256",
        FEATURE_MODULE_SHA256 == REQUIRED_FEATURE_MODULE_SHA256,
        FEATURE_MODULE_SHA256[:16] + "...")
require("feature module matches its own manifest record",
        FEATURE_MODULE_SHA256 == str(FEATURE_MANIFEST["module"]["sha256"]), "agrees")
require("loader module matches the frozen sha256",
        LOADER_MODULE_SHA256 == REQUIRED_LOADER_MODULE_SHA256,
        LOADER_MODULE_SHA256[:16] + "...")
require("loader module matches its own manifest record",
        LOADER_MODULE_SHA256 == str(LOADER_MANIFEST["module"]["sha256"]), "agrees")

for directory in (NB9_DIR, NB8_DIR):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
for name in ("bsl_rf_features", "bsl_sequence_loader"):
    sys.modules.pop(name, None)
import bsl_rf_features as R
import bsl_sequence_loader as L
print(f"\nimported {R.__file__}  ({R.FEATURE_SPEC_VERSION})")
print(f"imported {L.__file__}  ({L.LOADER_VERSION})")

# ---- the frozen ids --------------------------------------------------------
print("\nFROZEN IDS")
FOUND_IDS = {
    "experiment_id": str(FEATURE_MANIFEST["feature_contract"]["upstream"]["experiment_id"]),
    "loader_contract_id": str(LOADER_MANIFEST["loader_contract_id"]),
    "feature_contract_id": str(FEATURE_MANIFEST["feature_contract_id"]),
    "filter_set_id": str(FEATURE_MANIFEST["filter_set_id"]),
    "time_metadata_id": str(FEATURE_MANIFEST["time_metadata_id"]),
}
REQUIRED_IDS = {
    "experiment_id": REQUIRED_EXPERIMENT_ID,
    "loader_contract_id": REQUIRED_LOADER_CONTRACT_ID,
    "feature_contract_id": REQUIRED_FEATURE_CONTRACT_ID,
    "filter_set_id": REQUIRED_FILTER_SET_ID,
    "time_metadata_id": REQUIRED_TIME_METADATA_ID,
}
for name, wanted in REQUIRED_IDS.items():
    require(f"{name} matches the frozen value", FOUND_IDS[name] == wanted,
            FOUND_IDS[name])
require("the notebook 9 contract id reproduces from its own manifest",
        R.feature_manifest_id(FEATURE_MANIFEST["feature_contract"])
        == REQUIRED_FEATURE_CONTRACT_ID, REQUIRED_FEATURE_CONTRACT_ID)
require("the notebook 9 filter set id reproduces from its own digests",
        R.feature_manifest_id(FEATURE_MANIFEST["filter_digests"])
        == REQUIRED_FILTER_SET_ID, REQUIRED_FILTER_SET_ID)
require("the notebook 8 contract id reproduces from its own manifest",
        L.contract_id(LOADER_MANIFEST["loader_contract"])
        == REQUIRED_LOADER_CONTRACT_ID, REQUIRED_LOADER_CONTRACT_ID)
require("notebook 9 recorded a clean validation run",
        json.loads((NB9_DIR / "rf_feature_validation_report.json").read_text(
            encoding="utf-8"))["checks_failed"] == 0, "0 failed")

# ---- the frozen notebook 7 tables, verified by hash -----------------------
print("\nFROZEN FOLDS (read from the notebook 7 tables, verified by hash)")
manifest_table, manifest_strategy, _ = L.read_frozen_csv(
    NB7_DIR / "master_manifest.csv", L.MANIFEST_HASH_COLUMNS,
    L.REQUIRED_HASHES["manifest"], L.MANIFEST_TEXT_COLUMNS)
fold_table, fold_strategy, _ = L.read_frozen_csv(
    NB7_DIR / "fold_assignments.csv", L.FOLD_HASH_COLUMNS,
    L.REQUIRED_HASHES["fold_assignments"], L.FOLD_TEXT_COLUMNS)
control_table, control_strategy, _ = L.read_frozen_csv(
    NB7_DIR / "CONTROL_signer_overlap_split.csv", L.CONTROL_HASH_COLUMNS,
    L.REQUIRED_HASHES["control_split"], L.CONTROL_TEXT_COLUMNS)
for column in ("clip_id", "signer_id", "split"):
    fold_table[column] = fold_table[column].astype(str)
for column in ("clip_id", "split"):
    control_table[column] = control_table[column].astype(str)
manifest_table["clip_id"] = manifest_table["clip_id"].astype(str)
manifest_table["signer_id"] = manifest_table["signer_id"].astype(str)
print(f"  csv read strategy: manifest={manifest_strategy}, folds={fold_strategy}, "
      f"control={control_strategy}")

label_map = json.loads((NB7_DIR / "label_map.json").read_text(encoding="utf-8"))
LABEL_TO_ID = {str(k): int(v) for k, v in label_map["label_to_id"].items()}
experiment = json.loads((NB7_DIR / "experiment_manifest.json").read_text(encoding="utf-8"))
rebuilt_experiment_id = L.compute_experiment_id({
    "clip_table": L.REQUIRED_CLIP_TABLE_HASH,
    "manifest": L.canonical_hash(manifest_table, L.MANIFEST_HASH_COLUMNS),
    "fold_assignments": L.canonical_hash(fold_table, L.FOLD_HASH_COLUMNS),
    "signer_pairs": L.json_hash([list(entry["signers"]) for entry in json.loads(
        (NB7_DIR / "signer_pairs.json").read_text(encoding="utf-8"))["pairs"]]),
    "pair_order": L.json_hash(list(experiment["pairing"]["final_cyclic_order"])),
    "control_split": L.canonical_hash(control_table, L.CONTROL_HASH_COLUMNS),
    "label_map": L.sha16(json.dumps(LABEL_TO_ID, sort_keys=True, separators=(",", ":"))),
    "manifest_version": str(experiment["manifest_version"]),
})
require("the experiment id rebuilds from the frozen notebook 7 tables",
        rebuilt_experiment_id == REQUIRED_EXPERIMENT_ID, rebuilt_experiment_id)

FOLDS = L.reconstruct_folds(fold_table)
print(f"{'fold':>4}  {'test':<10} {'validation':<10} {'train signers':<40}")
for fold in range(N_FOLDS):
    print(f"{fold:>4}  {' '.join(FOLDS[fold]['test']):<10} "
          f"{' '.join(FOLDS[fold]['validation']):<10} "
          f"{' '.join(FOLDS[fold]['train'])}")
require("eight folds rebuilt with 12 / 2 / 2 signers",
        len(FOLDS) == N_FOLDS
        and all(len(FOLDS[f]["train"]) == TRAIN_SIGNERS_PER_FOLD
                and len(FOLDS[f]["validation"]) == 2 and len(FOLDS[f]["test"]) == 2
                for f in FOLDS), "8 of 8")

CLIP_META = manifest_table.set_index("clip_id")
LABEL_OF = CLIP_META["label_id"].astype(int)
SIGNER_OF = CLIP_META["signer_id"].astype(str)
print(f"\nmanifest rows {len(manifest_table):,} | labels "
      f"{manifest_table['label_id'].nunique()} | signers "
      f"{manifest_table['signer_id'].nunique()}")
require("the frozen manifest holds 6,000 clips, 200 labels, 16 signers",
        len(manifest_table) == N_CLIPS
        and manifest_table["label_id"].nunique() == N_LABELS
        and manifest_table["signer_id"].nunique() == N_SIGNERS, "as frozen")

In [ ]:
# ============================================================
# CELL 3 — Feature matrices, frozen filters, row index, FPS regimes
# ============================================================

print("CLIP ORDER")
clip_order = pd.read_csv(NB9_DIR / "rf_clip_order.csv", dtype={"clip_id": str})
CLIP_IDS: List[str] = clip_order["clip_id"].tolist()
ROW_OF = {clip_id: position for position, clip_id in enumerate(CLIP_IDS)}
require("clip order holds 6,000 unique clips in sorted order",
        len(CLIP_IDS) == N_CLIPS and len(set(CLIP_IDS)) == N_CLIPS
        and CLIP_IDS == sorted(CLIP_IDS), f"{len(CLIP_IDS):,}")
require("clip order matches the frozen notebook 7 manifest exactly",
        set(CLIP_IDS) == set(manifest_table["clip_id"]), "same 6,000 clips")
require("the row index is contiguous",
        clip_order["row"].tolist() == list(range(N_CLIPS)), f"0..{N_CLIPS - 1}")

Y_ALL = LABEL_OF.loc[CLIP_IDS].to_numpy(dtype=np.int64)
SIGNER_ALL = SIGNER_OF.loc[CLIP_IDS].to_numpy(dtype=object)
require("labels cover 0..199 exactly",
        sorted(np.unique(Y_ALL).tolist()) == list(range(N_LABELS)), "0..199")

# ---- feature specs, matrices --------------------------------------------
print("\nFEATURE SPECS AND MATRICES")
SPEC: Dict[str, pd.DataFrame] = {}
MATRIX: Dict[str, np.ndarray] = {}
for branch in BRANCHES:
    SPEC[branch] = pd.read_csv(NB9_DIR / f"rf_feature_spec_{branch}.csv")
    require(f"{branch} spec reproduces its frozen hash",
            R.spec_hash(SPEC[branch]) == REQUIRED_SPEC_HASH[branch],
            REQUIRED_SPEC_HASH[branch])
    require(f"{branch} spec has one row per feature",
            len(SPEC[branch]) == FEATURE_COUNT[branch], f"{len(SPEC[branch]):,}")

    MATRIX[branch] = np.load(NB9_DIR / f"rf_features_{branch}.npy", mmap_mode="r")
    print(f"  {branch:<8} matrix {MATRIX[branch].shape} {MATRIX[branch].dtype}  "
          f"{Path(NB9_DIR / f'rf_features_{branch}.npy').stat().st_size / 1e6:.0f} MB "
          "(mmap)")
    require(f"{branch} matrix has the frozen shape and dtype",
            tuple(MATRIX[branch].shape) == (N_CLIPS, FEATURE_COUNT[branch])
            and MATRIX[branch].dtype == np.float32, str(MATRIX[branch].shape))
    digest = array_digest(np.asarray(MATRIX[branch]))
    require(f"{branch} matrix reproduces its frozen digest",
            digest == REQUIRED_MATRIX_DIGEST[branch], digest)
    gc.collect()

# ---- frozen filters --------------------------------------------------------
print("\nFROZEN ZERO-VARIANCE FILTERS (loaded, never recomputed)")
FILTERS: Dict[Tuple[str, Any], Any] = {}
KEEP_IDX: Dict[Tuple[str, Any], np.ndarray] = {}
for branch in BRANCHES:
    fold_filters = R.load_filters(NB9_DIR / f"fold_feature_filters_{branch}.npz")
    control_filters = R.load_filters(NB9_DIR / f"control_feature_filter_{branch}.npz")
    for fold in range(N_FOLDS):
        FILTERS[(branch, fold)] = R.require_filter_scope(
            fold_filters[f"fold_{fold}"], branch, "fold", fold)
    FILTERS[(branch, "control")] = R.require_filter_scope(
        control_filters["control"], branch, "control")

    line = []
    for scope in list(range(N_FOLDS)) + ["control"]:
        fitted = FILTERS[(branch, scope)]
        KEEP_IDX[(branch, scope)] = np.flatnonzero(fitted.keep)
        expected = EXPECTED_RETAINED[branch][scope]
        require(f"{branch} {scope} retained dimension is exactly {expected:,}",
                fitted.kept == expected == int(KEEP_IDX[(branch, scope)].size),
                f"{fitted.kept:,}")
        line.append(f"{scope}:{fitted.kept:,}")
    print(f"  {branch:<8} " + "  ".join(line))

require("every filter digest matches the frozen filter set",
        R.feature_manifest_id({
            "folds": {b: {f"fold_{f}": FILTERS[(b, f)].digest() for f in range(N_FOLDS)}
                      for b in BRANCHES},
            "control": {b: FILTERS[(b, "control")].digest() for b in BRANCHES},
        }) == REQUIRED_FILTER_SET_ID, REQUIRED_FILTER_SET_ID)

# ---- row index per scope ---------------------------------------------------
FOLD_ROWS: Dict[Tuple[int, str], np.ndarray] = {}
for fold in range(N_FOLDS):
    for split in ("train", "validation", "test"):
        clips = sorted(fold_table.loc[(fold_table["fold"] == fold)
                                      & (fold_table["split"] == split), "clip_id"])
        FOLD_ROWS[(fold, split)] = np.array([ROW_OF[c] for c in clips], dtype=np.int64)
CONTROL_ROWS: Dict[str, np.ndarray] = {}
for split in ("train", "validation", "test"):
    clips = sorted(control_table.loc[control_table["split"] == split, "clip_id"])
    CONTROL_ROWS[split] = np.array([ROW_OF[c] for c in clips], dtype=np.int64)

print("\nSCOPE SIZES")
print(f"{'fold':>4}  {'train':>6} {'val':>5} {'test':>5}")
for fold in range(N_FOLDS):
    sizes = {s: FOLD_ROWS[(fold, s)].size for s in ("train", "validation", "test")}
    print(f"{fold:>4}  {sizes['train']:>6} {sizes['validation']:>5} {sizes['test']:>5}")
    require(f"fold {fold} splits partition the dataset exactly once",
            sum(sizes.values()) == N_CLIPS
            and len(set(np.concatenate([FOLD_ROWS[(fold, s)]
                                        for s in ("train", "validation", "test")]))) == N_CLIPS,
            sum(sizes.values()))
print(f"control  {CONTROL_ROWS['train'].size:>6} "
      f"{CONTROL_ROWS['validation'].size:>5} {CONTROL_ROWS['test'].size:>5}")
require("control split sizes are 4500 / 750 / 750",
        {s: int(CONTROL_ROWS[s].size) for s in CONTROL_ROWS} == CONTROL_SPLIT_SIZES,
        CONTROL_SPLIT_SIZES)

# ---- FPS regime diagnostic (descriptive only, never a predictor) -----------
print("\nFPS REGIME DIAGNOSTIC (descriptive metadata for notebook 13 only)")
fps_table = pd.read_csv(NB9_DIR / "rf_fps_by_clip.csv",
                        dtype={"clip_id": str, "fps_hex": str})
FPS = pd.Series({row.clip_id: float.fromhex(row.fps_hex)
                 for row in fps_table.itertuples()}).loc[CLIP_IDS]
require("the frozen fps table covers every clip exactly once",
        len(fps_table) == N_CLIPS and set(fps_table["clip_id"]) == set(CLIP_IDS),
        f"{len(fps_table):,}")
require("the fps mapping reproduces the frozen time metadata id",
        R.time_metadata_id({c: float(FPS[c]) for c in CLIP_IDS}, CLIP_IDS)
        == REQUIRED_TIME_METADATA_ID, REQUIRED_TIME_METADATA_ID)

REGIME = pd.Series(np.where(FPS.to_numpy() < FPS_REGIME_BOUNDARY,
                            REGIME_LABELS[0], REGIME_LABELS[1]), index=CLIP_IDS)
SIGNER_REGIME: Dict[str, str] = {}
for signer, group in pd.DataFrame({"signer": SIGNER_ALL,
                                   "regime": REGIME.to_numpy()}).groupby("signer"):
    present = sorted(set(group["regime"]))
    SIGNER_REGIME[str(signer)] = present[0] if len(present) == 1 else "mixed"
regime_counts = REGIME.value_counts().reindex(REGIME_LABELS).fillna(0).astype(int)
print(f"  clips per regime: " + ", ".join(f"{k} {v:,}" for k, v in regime_counts.items()))
print("  signer regimes  : " + ", ".join(
    f"{s}={SIGNER_REGIME[s]}" for s in sorted(SIGNER_REGIME,
                                              key=lambda v: int(v.lstrip('p')))))
require("the regime split reproduces the notebook 9 counts",
        int(regime_counts.iloc[0]) == 2365 and int(regime_counts.iloc[1]) == 3635,
        f"{int(regime_counts.iloc[0]):,} / {int(regime_counts.iloc[1]):,}")
print("  Recording regime is never used as a predictor, only reported.")

In [ ]:
# ============================================================
# CELL 4 — Freeze the RF model contract before any training
# ============================================================

RF_MODEL_CONTRACT = {
    "notebook": "10-random-forest-8-fold",
    "rf_policy_version": RF_POLICY_VERSION,
    "upstream": dict(REQUIRED_IDS,
                     feature_module_sha256=REQUIRED_FEATURE_MODULE_SHA256,
                     loader_module_sha256=REQUIRED_LOADER_MODULE_SHA256,
                     feature_spec_hash=dict(REQUIRED_SPEC_HASH),
                     feature_matrix_digest=dict(REQUIRED_MATRIX_DIGEST)),
    "estimator": "sklearn.ensemble.RandomForestClassifier",
    "sklearn_version": sklearn.__version__,
    "parameters": dict(RF_PARAMS),
    "parameters_note": "every parameter not listed keeps its sklearn default for the "
                       "recorded sklearn version",
    "max_features_primary": MAX_FEATURES_PRIMARY,
    "max_features_rationale": {
        "policy": "absolute candidate-search breadth is held equal between the full and "
                  "reduced branches at 96 features per split",
        "not_neutral": "this is NOT neutral relative to sklearn's sqrt convention. "
                       f"Reduced retains ~{EXPECTED_RETAINED['reduced'][2]:,} features, "
                       f"sqrt ~ {np.sqrt(EXPECTED_RETAINED['reduced'][2]):.0f}, so 96 is "
                       f"~{MAX_FEATURES_PRIMARY / np.sqrt(EXPECTED_RETAINED['reduced'][2]):.2f}x "
                       f"its natural breadth. Full retains "
                       f"~{EXPECTED_RETAINED['full'][2]:,} features, sqrt ~ "
                       f"{np.sqrt(EXPECTED_RETAINED['full'][2]):.0f}, so 96 is "
                       f"~{MAX_FEATURES_PRIMARY / np.sqrt(EXPECTED_RETAINED['full'][2]):.2f}x "
                       "its natural breadth.",
        "direction": "the primary setting is potentially conservative toward Full in "
                     "per-split search breadth relative to its natural sqrt setting, "
                     "and this direction must be stated wherever the primary RF "
                     "comparison is reported",
        "secondary_preregistration": "max_features='sqrt' is pre-registered as a "
                                     "SECONDARY RF sensitivity analysis only. It is not "
                                     "executed in this notebook and must never replace "
                                     "the max_features=96 primary result.",
        "frozen_before_results": True,
    },
    "dimensionality": {
        "raw_engineered_ratio": round(FEATURE_COUNT["full"] / FEATURE_COUNT["reduced"], 4),
        "raw_engineered": {b: FEATURE_COUNT[b] for b in BRANCHES},
        "post_zero_variance_ratio": round(
            EXPECTED_RETAINED["full"][2] / EXPECTED_RETAINED["reduced"][2], 4),
        "post_zero_variance_typical": {b: EXPECTED_RETAINED[b][2] for b in BRANCHES},
        "note": "the raw engineered ratio and the post-zero-variance model-input ratio "
                "are different quantities and are reported separately; the model-input "
                "ratio is what the forests actually saw",
    },
    "retained_dimensions": {b: {str(k): int(v)
                                for k, v in EXPECTED_RETAINED[b].items()}
                            for b in BRANCHES},
    "folds": {
        "source": "frozen notebook 7 fold_assignments.csv, verified by hash and rebuilt "
                  "with reconstruct_folds",
        "design": "12 train / 2 validation / 2 unseen test signers per fold",
        "training_rows": "the 12 training signers of the fold only",
        "validation_role": "DIAGNOSTIC ONLY. Validation predictions are saved but never "
                           "used for fitting, model selection or any decision in this "
                           "notebook.",
        "validation_rationale": "Random Forest requires no early stopping, but BiLSTM "
                                "and BiGRU will use these same two validation signers "
                                "for training control. Restricting RF training to the "
                                "same 12 training signers preserves cross-model "
                                "comparability.",
        "no_refit_on_train_plus_validation": True,
    },
    "seeds": {
        "fold": f"random_state = {FOLD_SEED_BASE} + fold",
        "control": f"random_state = {CONTROL_SEED}",
        "shared_across_branches": "Full and Reduced use the same random seed within "
                                  "each fold. This aligns the estimator-level random "
                                  "seed and helps control bootstrap-sampling randomness "
                                  "between branches. Feature-subset draws cannot be "
                                  "identical because the two branches have different "
                                  "feature spaces.",
    },
    "filters": {
        "source": "frozen notebook 9 zero-variance keep-masks, loaded and scope-checked",
        "recomputed": False,
        "transformations_applied": [],
        "not_applied": ["scaling", "standardisation", "PCA",
                        "any additional feature selection"],
    },
    "resource_gate": {
        "max_projected_forest_value_gib": MAX_PROJECTED_FOREST_VALUE_GIB,
        "probe": f"{BENCHMARK_ESTIMATORS}-tree forests on fold 0 TRAIN rows only, per "
                 "branch, with otherwise frozen settings",
        "projection": "mean nodes per probe tree x n_estimators x n_classes x 8 bytes, "
                     "the size of the sklearn per-node class-count arrays",
        "action_on_breach": "STOP before the primary training loop. No validation or "
                            "test prediction is generated, so n_estimators can be "
                            "reconsidered for resource reasons without any evaluation "
                            "result having been seen.",
        "automatic_adjustment": False,
        "rationale": "with 200 classes each tree stores a (n_nodes, 1, 200) float64 "
                     "value array, so forest memory is the binding constraint rather "
                     "than fit time",
    },
    "probability_class_order": "columns are class ids 0..199 in ascending order; "
                              "model.classes_ is asserted equal to arange(200) after "
                              "every fit",
    "predictions_from": "argmax over predict_proba, mapped through model.classes_",
    "metrics": {
        "per_fold": ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy",
                     f"top_{TOP_K}_accuracy", "per_test_signer_accuracy",
                     "signer_macro_accuracy"],
        "pooled_oof": ["accuracy", "macro_f1", "weighted_f1", "balanced_accuracy",
                       f"top_{TOP_K}_accuracy", "signer_macro_accuracy",
                       "per_signer_accuracy", "per_class_recall",
                       "confusion_matrix_200x200"],
        "validation": "saved separately and labelled DIAGNOSTIC ONLY",
        "deferred_to_notebook_13": ["paired sign-flip test", "confidence intervals",
                                    "any equivalence or significance claim"],
    },
    "oof_policy": {
        "definition": "each of the 6,000 clips receives exactly one prediction, from the "
                      "fold in which its signer is a TEST signer",
        "verified": ["exactly 6,000 rows", "every clip exactly once",
                     "no missing or duplicated clip", "probability columns 0..199",
                     "finite probabilities", "row sums approximately 1"],
        "probability_dtype": "float32",
    },
    "control_policy": {
        "runs": "exactly one signer-overlap control per branch",
        "splits": dict(CONTROL_SPLIT_SIZES),
        "label": "INTENTIONAL SIGNER-OVERLAP CONTROL",
        "separation": "control predictions are never mixed with the 6,000 "
                      "signer-independent out-of-fold results",
        "filter": "the frozen notebook 9 control keep-mask only",
    },
    "feature_importance": {
        "kind": "sklearn impurity-based feature_importances_",
        "status": "EXPLORATORY",
        "expanded_to": {b: [N_FOLDS, FEATURE_COUNT[b]] for b in BRANCHES},
        "filtered_out_features": "recorded as 0.0",
        "permutation_importance": False,
    },
    "resume_rules": {
        "scopes": N_FOLDS * len(BRANCHES) + len(BRANCHES),
        "overwrite_default": OVERWRITE,
        "artefact_written": "immediately after each scope completes, atomically via a "
                           "temporary file and rename",
        "skip_requires": ["model contract id matches", "upstream provenance matches",
                          "expected row counts match",
                          "prediction digest reproduces from the saved arrays",
                          "probability shape and class order valid"],
        "on_mismatch": "recompute the scope; stale partial output is never silently "
                       "reused",
    },
    "regime_diagnostic": {
        "source": "frozen notebook 9 fps table",
        "boundary": f"fps < {FPS_REGIME_BOUNDARY} is '{REGIME_LABELS[0]}', otherwise "
                    f"'{REGIME_LABELS[1]}'",
        "use": "descriptive columns in rf_fold_metrics.csv for notebook 13 only",
        "never_a_predictor": True,
    },
    "forbidden": ["regenerating folds, features or filters",
                  "tuning on validation or test results",
                  "changing max_features after seeing results",
                  "retraining on train + validation",
                  "mixing control and out-of-fold predictions",
                  "persisting the 18 trained forests"],
}
RF_MODEL_CONTRACT_ID = sha16(json.dumps(RF_MODEL_CONTRACT, sort_keys=True,
                                        separators=(",", ":"), default=str))

print("RF MODEL CONTRACT")
print(f"  rf model contract id : {RF_MODEL_CONTRACT_ID}")
print(f"  sklearn              : {sklearn.__version__}")
print(f"  estimator            : {N_ESTIMATORS} trees, max_features="
      f"{MAX_FEATURES_PRIMARY}, gini, unlimited depth, bootstrap")

blob = json.dumps(RF_MODEL_CONTRACT, default=str)
require("the contract carries no session path", "/kaggle" not in blob, "stable")
require("the contract carries no timestamp",
        "generated_at" not in blob and not any(
            token in blob for token in ("T00:", "T12:")), "stable")
require("the contract records the max_features direction explicitly",
        "potentially conservative toward Full"
        in RF_MODEL_CONTRACT["max_features_rationale"]["direction"], "recorded")
require("the contract pre-registers sqrt as secondary only",
        "must never replace"
        in RF_MODEL_CONTRACT["max_features_rationale"]["secondary_preregistration"],
        "recorded")
require("the contract records both dimensionality ratios",
        RF_MODEL_CONTRACT["dimensionality"]["raw_engineered_ratio"] == 2.7964
        and RF_MODEL_CONTRACT["dimensionality"]["post_zero_variance_ratio"] == 2.8138,
        f"{RF_MODEL_CONTRACT['dimensionality']['raw_engineered_ratio']} and "
        f"{RF_MODEL_CONTRACT['dimensionality']['post_zero_variance_ratio']}")
require("the contract records why validation is excluded from RF training",
        "cross-model comparability"
        in RF_MODEL_CONTRACT["folds"]["validation_rationale"], "recorded")
require("the contract records the shared-seed policy",
        "same random seed within" in RF_MODEL_CONTRACT["seeds"]["shared_across_branches"],
        "recorded")
require("the contract does not overclaim feature-subset alignment",
        "cannot be identical"
        in RF_MODEL_CONTRACT["seeds"]["shared_across_branches"]
        and "feature-sampling randomness align"
        not in json.dumps(RF_MODEL_CONTRACT), "different feature spaces acknowledged")
require("the contract records the resource gate",
        RF_MODEL_CONTRACT["resource_gate"]["max_projected_forest_value_gib"]
        == MAX_PROJECTED_FOREST_VALUE_GIB
        and RF_MODEL_CONTRACT["resource_gate"]["automatic_adjustment"] is False,
        f"{MAX_PROJECTED_FOREST_VALUE_GIB} GiB, no automatic change")

atomic_write(OUT["contract"], lambda h: h.write(
    json.dumps(RF_MODEL_CONTRACT, indent=2, sort_keys=True, default=str).encode()))
reloaded_contract = json.loads(OUT["contract"].read_text(encoding="utf-8"))
require("the rf model contract id reproduces from the file on disk",
        sha16(json.dumps(reloaded_contract, sort_keys=True, separators=(",", ":"),
                         default=str)) == RF_MODEL_CONTRACT_ID, RF_MODEL_CONTRACT_ID)
print(f"  wrote {OUT['contract'].name}  {OUT['contract'].stat().st_size:,} bytes")

In [ ]:
# ============================================================
# CELL 5 — Scope helpers: safe materialisation, metrics, resume artefacts
# ============================================================
# Memory matters more than CPU here. Rows and retained columns are selected in a
# SINGLE allocation with np.ix_, never as X[rows][:, keep], which would build a
# full-width intermediate first.

ALL_SCOPES: List[Tuple[str, Any]] = [(b, f) for f in range(N_FOLDS) for b in BRANCHES]
ALL_SCOPES += [(b, "control") for b in BRANCHES]


def scope_rows(scope: Any) -> Dict[str, np.ndarray]:
    if scope == "control":
        return {split: CONTROL_ROWS[split] for split in ("train", "validation", "test")}
    return {split: FOLD_ROWS[(scope, split)]
            for split in ("train", "validation", "test")}


def materialise(branch: str, scope: Any, rows: np.ndarray) -> np.ndarray:
    """One allocation: (len(rows), retained) float32."""
    keep_idx = KEEP_IDX[(branch, scope)]
    block = MATRIX[branch][np.ix_(np.sort(rows), keep_idx)]
    return np.ascontiguousarray(block, dtype=np.float32)


def verify_scope(branch: str, scope: Any, rows: Dict[str, np.ndarray]) -> None:
    """Everything that must hold before a single tree is grown."""
    label = scope_label(branch, scope)
    train_clips = [CLIP_IDS[i] for i in rows["train"]]
    y_train = Y_ALL[np.sort(rows["train"])]

    if scope != "control":
        definition = FOLDS[scope]
        train_signers = set(SIGNER_ALL[np.sort(rows["train"])])
        validation_signers = set(SIGNER_ALL[np.sort(rows["validation"])])
        test_signers = set(SIGNER_ALL[np.sort(rows["test"])])
        require(f"{label}: train / validation / test signers disjoint",
                not (train_signers & validation_signers)
                and not (train_signers & test_signers)
                and not (validation_signers & test_signers),
                f"{len(train_signers)}/{len(validation_signers)}/{len(test_signers)}")
        require(f"{label}: training signers are exactly the frozen 12",
                train_signers == set(definition["train"]), len(train_signers))
        require(f"{label}: training rows match the frozen fold exactly",
                sorted(train_clips) == sorted(
                    fold_table.loc[(fold_table["fold"] == scope)
                                   & (fold_table["split"] == "train"), "clip_id"]),
                f"{len(train_clips):,} clips")
        require(f"{label}: all 200 classes present in validation and test",
                len(np.unique(Y_ALL[np.sort(rows["validation"])])) == N_LABELS
                and len(np.unique(Y_ALL[np.sort(rows["test"])])) == N_LABELS,
                "200 / 200")
    else:
        overlap = set(SIGNER_ALL[np.sort(rows["train"])]) & set(
            SIGNER_ALL[np.sort(rows["test"])])
        require(f"{label}: signer overlap is present by design",
                len(overlap) == N_SIGNERS, f"{len(overlap)} of {N_SIGNERS} signers")

    require(f"{label}: all 200 classes present in training",
            len(np.unique(y_train)) == N_LABELS, len(np.unique(y_train)))
    require(f"{label}: training labels are exactly 0..199",
            np.array_equal(np.unique(y_train), np.arange(N_LABELS)), "0..199")
    require(f"{label}: no validation or test row appears in training",
            not (set(rows["train"].tolist()) & set(rows["validation"].tolist()))
            and not (set(rows["train"].tolist()) & set(rows["test"].tolist())),
            "disjoint rows")


def top_k_accuracy(proba: np.ndarray, y_true: np.ndarray, k: int = TOP_K) -> float:
    top = np.argpartition(-proba, kth=k - 1, axis=1)[:, :k]
    return float(np.mean([y_true[i] in top[i] for i in range(y_true.size)]))


def scope_metrics(y_true: np.ndarray, y_pred: np.ndarray, proba: np.ndarray,
                  signers: np.ndarray) -> Dict[str, Any]:
    per_signer = {}
    for signer in sorted(set(signers.tolist()), key=lambda v: int(str(v).lstrip("p"))):
        mask = signers == signer
        per_signer[str(signer)] = float(accuracy_score(y_true[mask], y_pred[mask]))
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        f"top{TOP_K}_accuracy": top_k_accuracy(proba, y_true),
        "signer_macro_accuracy": float(np.mean(list(per_signer.values()))),
        "per_signer_accuracy": per_signer,
        "n": int(y_true.size),
    }


def probabilities_valid(proba: np.ndarray, rows: int) -> bool:
    return (proba.shape == (rows, N_LABELS) and proba.dtype == np.float32
            and bool(np.isfinite(proba).all()) and float(proba.min()) >= 0.0
            and float(np.abs(proba.sum(axis=1) - 1.0).max()) < 1e-3)


SCOPE_PROVENANCE = dict(REQUIRED_IDS, rf_model_contract_id=RF_MODEL_CONTRACT_ID)


def scope_path(branch: str, scope: Any) -> Path:
    return SCOPE_DIR / f"rf_scope_{scope_label(branch, scope)}.npz"


def save_scope(branch: str, scope: Any, payload: Dict[str, Any]) -> Path:
    path = scope_path(branch, scope)
    arrays = {key: value for key, value in payload.items()
              if isinstance(value, np.ndarray)}
    meta = {key: value for key, value in payload.items()
            if not isinstance(value, np.ndarray)}
    arrays["__meta__"] = np.array([json.dumps(meta, sort_keys=True, default=str)])
    temporary = path.parent / (path.name + ".tmp")
    with open(temporary, "wb") as handle:
        np.savez_compressed(handle, **arrays)
    # Validate the temporary file before it is allowed to become the artefact.
    with np.load(temporary, allow_pickle=False) as check:
        reloaded_meta = json.loads(str(check["__meta__"][0]))
        ok = (reloaded_meta["prediction_digest"]
              == array_digest(check["y_pred"], check["proba"])
              and probabilities_valid(check["proba"], int(check["y_true"].size)))
    if not ok:
        os.unlink(temporary)
        raise RuntimeError(f"{path.name} failed post-write validation; not published")
    os.replace(temporary, path)
    return path


def load_scope(branch: str, scope: Any) -> Optional[Dict[str, Any]]:
    """Return a completed scope only if it is fully trustworthy."""
    path = scope_path(branch, scope)
    if OVERWRITE or not path.exists():
        return None
    try:
        with np.load(path, allow_pickle=False) as archive:
            meta = json.loads(str(archive["__meta__"][0]))
            arrays = {key: np.array(archive[key]) for key in archive.files
                      if key != "__meta__"}
    except Exception as error:
        print(f"    resume rejected ({type(error).__name__}); recomputing")
        return None

    rows = scope_rows(scope)
    reasons = []
    if meta.get("rf_model_contract_id") != RF_MODEL_CONTRACT_ID:
        reasons.append("contract id")
    if {k: meta.get(k) for k in REQUIRED_IDS} != REQUIRED_IDS:
        reasons.append("provenance")
    if meta.get("branch") != branch or str(meta.get("scope")) != str(scope):
        reasons.append("scope label")
    if int(meta.get("n_test", -1)) != int(rows["test"].size):
        reasons.append("test rows")
    if int(meta.get("n_train", -1)) != int(rows["train"].size):
        reasons.append("train rows")
    if int(meta.get("n_features", -1)) != int(KEEP_IDX[(branch, scope)].size):
        reasons.append("retained dimension")
    if meta.get("prediction_digest") != array_digest(arrays["y_pred"], arrays["proba"]):
        reasons.append("prediction digest")
    if not probabilities_valid(arrays["proba"], int(rows["test"].size)):
        reasons.append("probability validity")
    if reasons:
        print(f"    resume rejected ({', '.join(reasons)}); recomputing")
        return None
    return {**meta, **arrays}


completed = sum(1 for branch, scope in ALL_SCOPES
                if scope_path(branch, scope).exists())
print(f"scopes to run   : {len(ALL_SCOPES)}")
print(f"artefacts present: {completed} (each is re-verified before being reused)")
print(f"OVERWRITE       : {OVERWRITE}")

In [ ]:
# ============================================================
# CELL 6 — Resource preflight (timing only, never a tuning signal)
# ============================================================
# Fold 0 training rows only, 25 trees, otherwise the frozen settings. No test data
# is touched and no hyperparameter is changed on the basis of what this measures.

print(f"PREFLIGHT  fold 0 train, {BENCHMARK_ESTIMATORS} trees, "
      f"max_features={MAX_FEATURES_PRIMARY}")
benchmark_rows = np.sort(FOLD_ROWS[(0, "train")])
y_benchmark = Y_ALL[benchmark_rows]
PREFLIGHT: List[Dict[str, Any]] = []

for branch in BRANCHES:
    baseline_memory = peak_memory_gib()
    started = time.time()
    X_benchmark = materialise(branch, 0, benchmark_rows)
    materialise_seconds = time.time() - started
    array_gib = X_benchmark.nbytes / (1024 ** 3)

    started = time.time()
    probe = RandomForestClassifier(**{**RF_PARAMS,
                                      "n_estimators": BENCHMARK_ESTIMATORS},
                                   random_state=scope_seed(0))
    probe.fit(X_benchmark, y_benchmark)
    fit_seconds = time.time() - started
    nodes = int(sum(tree.tree_.node_count for tree in probe.estimators_))
    peak_after = peak_memory_gib()

    projected_fit = fit_seconds * N_ESTIMATORS / BENCHMARK_ESTIMATORS
    projected_forest_gib = (nodes / BENCHMARK_ESTIMATORS * N_ESTIMATORS
                            * N_LABELS * 8) / (1024 ** 3)
    PREFLIGHT.append({
        "branch": branch, "rows": int(X_benchmark.shape[0]),
        "features": int(X_benchmark.shape[1]),
        "materialise_seconds": round(materialise_seconds, 2),
        "train_array_gib": round(array_gib, 3),
        "benchmark_trees": BENCHMARK_ESTIMATORS,
        "benchmark_fit_seconds": round(fit_seconds, 2),
        "mean_nodes_per_tree": int(nodes / BENCHMARK_ESTIMATORS),
        "projected_fit_seconds_300": round(projected_fit, 1),
        "projected_forest_value_gib": round(projected_forest_gib, 6),
        "observed_peak_rss_gib": round(peak_after, 2),
        "peak_rss_delta_gib": round(peak_after - baseline_memory, 2),
    })
    print(f"  {branch:<8} {X_benchmark.shape[0]:,} x {X_benchmark.shape[1]:,} "
          f"= {array_gib:.2f} GiB | materialise {materialise_seconds:.1f}s | "
          f"{BENCHMARK_ESTIMATORS} trees {fit_seconds:.1f}s")
    print(f"           mean {int(nodes / BENCHMARK_ESTIMATORS):,} nodes/tree | "
          f"projected {N_ESTIMATORS} trees ~{projected_fit / 60:.1f} min | "
          f"forest value arrays ~{projected_forest_gib:.2f} GiB | "
          f"peak RSS {peak_after:.1f} GiB")

    require(f"{branch} benchmark used the frozen retained dimension",
            int(X_benchmark.shape[1]) == EXPECTED_RETAINED[branch][0],
            f"{X_benchmark.shape[1]:,}")
    require(f"{branch} benchmark forest learned all 200 classes",
            np.array_equal(probe.classes_, np.arange(N_LABELS)), "0..199")
    del probe, X_benchmark
    gc.collect()

preflight_table = pd.DataFrame(PREFLIGHT)
projected_total = float(preflight_table["projected_fit_seconds_300"].sum())
# 8 folds per branch plus one control per branch, using each branch's own projection.
projected_all = projected_total * (N_FOLDS + 1)
print(f"\n  projected primary + control fitting: ~{projected_all / 60:.0f} min "
      f"({N_FOLDS + 1} scopes per branch)")
require("preflight measured both branches", len(PREFLIGHT) == len(BRANCHES),
        f"{len(PREFLIGHT)} of {len(BRANCHES)}")
require("preflight used fold 0 training rows only", True,
        "no validation or test row was touched")

# ============================================================
# RESOURCE GATE — a real stop, not a warning
# ============================================================
# The point of the gate is that n_estimators can still be reconsidered for resource
# reasons at this moment, because no validation or test prediction exists yet. If the
# projection breaches the ceiling the notebook stops here; nothing is adjusted
# automatically.
worst_projected = float(preflight_table["projected_forest_value_gib"].max())
print(f"\nRESOURCE GATE  ceiling {MAX_PROJECTED_FOREST_VALUE_GIB} GiB "
      f"of forest value arrays at {N_ESTIMATORS} trees")
for row in PREFLIGHT:
    print(f"  {row['branch']:<8} {row['mean_nodes_per_tree']:>7,} nodes/tree  ->  "
          f"projected {row['projected_forest_value_gib']:.3f} GiB  "
          f"(train array {row['train_array_gib']:.2f} GiB, "
          f"peak RSS so far {row['observed_peak_rss_gib']:.2f} GiB)")
print(f"  worst projected      : {worst_projected:.2f} GiB")
print(f"  ceiling              : {MAX_PROJECTED_FOREST_VALUE_GIB:.2f} GiB")

PREFLIGHT_REPORT = {
    "notebook": "10-random-forest-8-fold", "stage": "resource_gate",
    "rf_model_contract_id": RF_MODEL_CONTRACT_ID,
    "upstream": dict(REQUIRED_IDS),
    "n_estimators": N_ESTIMATORS, "benchmark_estimators": BENCHMARK_ESTIMATORS,
    "max_features": MAX_FEATURES_PRIMARY,
    "max_projected_forest_value_gib": MAX_PROJECTED_FOREST_VALUE_GIB,
    "worst_projected_forest_value_gib": round(worst_projected, 6),
    "per_branch": PREFLIGHT,
    "projected_all_scopes_minutes": round(projected_all / 60, 1),
    "verdict": "PASS" if worst_projected <= MAX_PROJECTED_FOREST_VALUE_GIB
               else "STOP",
    "probe_scope": "fold 0 TRAIN rows only; no validation or test data was touched",
}
atomic_write(OUT["preflight_report"], lambda h: h.write(
    json.dumps(PREFLIGHT_REPORT, indent=2, default=str).encode()))
print(f"  wrote {OUT['preflight_report'].name}")

if worst_projected > MAX_PROJECTED_FOREST_VALUE_GIB:
    record("resource gate: projected forest memory within the ceiling", False,
           f"{worst_projected:.2f} GiB > {MAX_PROJECTED_FOREST_VALUE_GIB:.2f} GiB")
    diagnostic = {
        "notebook": "10-random-forest-8-fold",
        "verdict": "STOPPED_AT_RESOURCE_GATE", "stage": "resource_gate",
        "rf_model_contract_id": RF_MODEL_CONTRACT_ID,
        "preflight": PREFLIGHT_REPORT, "checks": CHECKS,
        "runtime_seconds": round(time.time() - STARTED, 1),
    }
    atomic_write(OUT["report_json"], lambda h: h.write(
        json.dumps(diagnostic, indent=2, default=str).encode()))
    atomic_write(OUT["report_csv"], lambda h: pd.DataFrame(CHECKS)[
        ["check", "status", "passed", "detail"]].to_csv(h, index=False))
    print("\n" + "=" * 78)
    print("STOPPED AT THE RESOURCE GATE - NO MODEL TRAINED, NO PREDICTION MADE")
    print("=" * 78)
    for row in PREFLIGHT:
        print(f"  {row['branch']:<8} projected forest value arrays "
              f"{row['projected_forest_value_gib']:.3f} GiB at {N_ESTIMATORS} trees")
    print(f"  worst {worst_projected:.2f} GiB exceeds the frozen ceiling "
          f"{MAX_PROJECTED_FOREST_VALUE_GIB:.2f} GiB.")
    print("  Everything above came from fold 0 TRAINING rows only. No validation or")
    print("  test prediction exists, so n_estimators may still be reconsidered for")
    print("  resource reasons without any evaluation result having been seen.")
    print("  This notebook does not adjust n_estimators. Decide the new value, record")
    print("  it as a frozen policy change, and rerun.")
    print("=" * 78)
    raise RuntimeError(
        f"Resource gate STOP: projected forest value arrays {worst_projected:.2f} GiB "
        f"exceed the frozen ceiling {MAX_PROJECTED_FOREST_VALUE_GIB:.2f} GiB. No "
        "training was started and no validation or test prediction was generated.")

require("resource gate: projected forest memory within the ceiling", True,
        f"{worst_projected:.2f} GiB <= {MAX_PROJECTED_FOREST_VALUE_GIB:.2f} GiB")
print("\n  Gate PASSED. Timing and memory only: no benchmark accuracy was computed")
print("  and no frozen hyperparameter was revised on the basis of this cell.")

In [ ]:
# ============================================================
# CELL 7 — Train the 18 scopes: 8 folds x 2 branches, plus 1 control per branch
# ============================================================
# One model in memory at a time. Everything is freed before the next scope starts,
# so full and reduced arrays never coexist.

MODEL_SIZE_SCOPES = {("full", 0), ("reduced", 0)}   # representative sizes only
SCOPE_RESULTS: Dict[Tuple[str, Any], Dict[str, Any]] = {}
EFFICIENCY: List[Dict[str, Any]] = []

for branch, scope in ALL_SCOPES:
    label = scope_label(branch, scope)
    is_control = scope == "control"
    print(f"\n{'=' * 74}\n{label.upper()}"
          + ("   [INTENTIONAL SIGNER-OVERLAP CONTROL]" if is_control else "")
          + f"\n{'=' * 74}")

    resumed = load_scope(branch, scope)
    if resumed is not None:
        SCOPE_RESULTS[(branch, scope)] = resumed
        print(f"  resumed from {scope_path(branch, scope).name}  "
              f"accuracy {resumed['metrics_test']['accuracy']:.4f}")
        record(f"{label}: resumed artefact passed every integrity check", True,
               resumed["prediction_digest"])
        EFFICIENCY.append(resumed["efficiency"])
        continue

    rows = scope_rows(scope)
    verify_scope(branch, scope, rows)
    seed = scope_seed(scope)
    keep_count = int(KEEP_IDX[(branch, scope)].size)
    require(f"{label}: retained dimension is the frozen value",
            keep_count == EXPECTED_RETAINED[branch][scope], f"{keep_count:,}")

    sorted_rows = {split: np.sort(rows[split]) for split in rows}
    started = time.time()
    X_train = materialise(branch, scope, sorted_rows["train"])
    y_train = Y_ALL[sorted_rows["train"]]
    materialise_seconds = time.time() - started

    model = RandomForestClassifier(**RF_PARAMS, random_state=seed)
    started = time.time()
    model.fit(X_train, y_train)
    fit_seconds = time.time() - started
    require(f"{label}: model.classes_ is exactly 0..199",
            np.array_equal(model.classes_, np.arange(N_LABELS)), "0..199")
    print(f"  trained {N_ESTIMATORS} trees on {X_train.shape[0]:,} x "
          f"{X_train.shape[1]:,} in {fit_seconds / 60:.1f} min (seed {seed})")

    importances = np.asarray(model.feature_importances_, dtype=np.float64)
    require(f"{label}: importances have the retained length",
            importances.shape == (keep_count,), importances.shape)

    predictions: Dict[str, Dict[str, Any]] = {}
    for split in ("validation", "test"):
        X_split = materialise(branch, scope, sorted_rows[split])
        started = time.time()
        proba = model.predict_proba(X_split).astype(np.float32)
        predict_seconds = time.time() - started
        y_true = Y_ALL[sorted_rows[split]]
        y_pred = model.classes_[np.argmax(proba, axis=1)].astype(np.int64)
        require(f"{label}: {split} probabilities are valid",
                probabilities_valid(proba, int(y_true.size)),
                f"{proba.shape}, row sums ~1")
        predictions[split] = {
            "clip_id": np.array([CLIP_IDS[i] for i in sorted_rows[split]]),
            "signer_id": SIGNER_ALL[sorted_rows[split]].astype(str),
            "y_true": y_true, "y_pred": y_pred, "proba": proba,
            "metrics": scope_metrics(y_true, y_pred, proba,
                                     SIGNER_ALL[sorted_rows[split]]),
            "predict_seconds": predict_seconds,
        }
        del X_split
        gc.collect()
    print(f"  test accuracy {predictions['test']['metrics']['accuracy']:.4f} | "
          f"macro F1 {predictions['test']['metrics']['macro_f1']:.4f} | "
          f"top{TOP_K} {predictions['test']['metrics'][f'top{TOP_K}_accuracy']:.4f}")
    print(f"  validation accuracy {predictions['validation']['metrics']['accuracy']:.4f}"
          "   [DIAGNOSTIC ONLY]")

    # Representative serialized size only, measured then deleted immediately.
    model_mib = None
    if (branch, scope) in MODEL_SIZE_SCOPES:
        import joblib
        probe_path = WORKING / f"__size_probe_{label}.joblib"
        joblib.dump(model, probe_path, compress=0)
        model_mib = probe_path.stat().st_size / (1024 ** 2)
        probe_path.unlink()
        print(f"  representative serialized model size {model_mib:,.0f} MiB "
              "(measured, file deleted, not a PASS/FAIL invariant)")

    total_nodes = int(sum(tree.tree_.node_count for tree in model.estimators_))
    efficiency = {
        "branch": branch, "scope": str(scope), "seed": seed,
        "train_rows": int(X_train.shape[0]), "features": keep_count,
        "materialise_seconds": round(materialise_seconds, 2),
        "fit_seconds": round(fit_seconds, 2),
        "predict_test_seconds": round(predictions["test"]["predict_seconds"], 3),
        "predict_validation_seconds": round(
            predictions["validation"]["predict_seconds"], 3),
        "total_tree_nodes": total_nodes,
        "mean_nodes_per_tree": int(total_nodes / N_ESTIMATORS),
        "train_array_gib": round(X_train.nbytes / (1024 ** 3), 3),
        "peak_rss_gib": round(peak_memory_gib(), 2),
        "serialized_model_mib": None if model_mib is None else round(model_mib, 1),
    }
    EFFICIENCY.append(efficiency)

    payload = {
        **SCOPE_PROVENANCE, "branch": branch, "scope": str(scope), "seed": seed,
        "is_control": bool(is_control), "n_train": int(rows["train"].size),
        "n_validation": int(rows["validation"].size), "n_test": int(rows["test"].size),
        "n_features": keep_count,
        "clip_id": predictions["test"]["clip_id"],
        "signer_id": predictions["test"]["signer_id"],
        "y_true": predictions["test"]["y_true"],
        "y_pred": predictions["test"]["y_pred"],
        "proba": predictions["test"]["proba"],
        "val_clip_id": predictions["validation"]["clip_id"],
        "val_signer_id": predictions["validation"]["signer_id"],
        "val_y_true": predictions["validation"]["y_true"],
        "val_y_pred": predictions["validation"]["y_pred"],
        "val_proba": predictions["validation"]["proba"],
        "importances": importances.astype(np.float64),
        "keep_idx": KEEP_IDX[(branch, scope)].astype(np.int64),
        "metrics_test": predictions["test"]["metrics"],
        "metrics_validation": predictions["validation"]["metrics"],
        "validation_status": "DIAGNOSTIC ONLY",
        "efficiency": efficiency,
        "prediction_digest": array_digest(predictions["test"]["y_pred"],
                                          predictions["test"]["proba"]),
    }
    written = save_scope(branch, scope, payload)
    SCOPE_RESULTS[(branch, scope)] = payload
    print(f"  wrote {written.name}  {written.stat().st_size / 1e6:.1f} MB")

    del model, X_train, y_train, predictions, importances
    gc.collect()

print(f"\n{'=' * 74}")
require("every scope produced a result", len(SCOPE_RESULTS) == len(ALL_SCOPES),
        f"{len(SCOPE_RESULTS)} of {len(ALL_SCOPES)}")
require("full and reduced used identical RF settings within every fold",
        all(SCOPE_RESULTS[("full", s)]["seed"] == SCOPE_RESULTS[("reduced", s)]["seed"]
            for s in list(range(N_FOLDS)) + ["control"]), "same seed per scope")
require("fold seeds follow the frozen policy",
        all(SCOPE_RESULTS[(b, f)]["seed"] == FOLD_SEED_BASE + f
            for b in BRANCHES for f in range(N_FOLDS))
        and all(SCOPE_RESULTS[(b, "control")]["seed"] == CONTROL_SEED
                for b in BRANCHES), f"{FOLD_SEED_BASE} + k, control {CONTROL_SEED}")
require("every saved prediction digest round-trips from disk",
        all(load_scope(b, s) is not None for b, s in ALL_SCOPES),
        f"{len(ALL_SCOPES)} artefacts")

In [ ]:
# ============================================================
# CELL 8 — Out-of-fold assembly: exactly one prediction per clip per branch
# ============================================================
# Each clip is predicted by the single fold in which its signer is a TEST signer.
# Control predictions are never mixed in here.

OOF: Dict[str, Dict[str, np.ndarray]] = {}
for branch in BRANCHES:
    parts = []
    for fold in range(N_FOLDS):
        scope = SCOPE_RESULTS[(branch, fold)]
        parts.append(pd.DataFrame({
            "clip_id": scope["clip_id"].astype(str),
            "signer_id": scope["signer_id"].astype(str),
            "label_id": scope["y_true"].astype(np.int64),
            "fold": np.full(scope["y_true"].size, fold, dtype=np.int64),
            "y_true": scope["y_true"].astype(np.int64),
            "y_pred": scope["y_pred"].astype(np.int64),
            "__row__": np.arange(scope["y_true"].size),
        }))
    table = pd.concat(parts, ignore_index=True)
    proba = np.concatenate([SCOPE_RESULTS[(branch, f)]["proba"]
                            for f in range(N_FOLDS)]).astype(np.float32)

    order = np.argsort(table["clip_id"].to_numpy(), kind="mergesort")
    table = table.iloc[order].reset_index(drop=True)
    proba = proba[order]

    OOF[branch] = {
        "clip_id": table["clip_id"].to_numpy().astype(str),
        "signer_id": table["signer_id"].to_numpy().astype(str),
        "label_id": table["label_id"].to_numpy(dtype=np.int64),
        "fold": table["fold"].to_numpy(dtype=np.int64),
        "y_true": table["y_true"].to_numpy(dtype=np.int64),
        "y_pred": table["y_pred"].to_numpy(dtype=np.int64),
        "proba": proba,
    }

print("OUT-OF-FOLD COVERAGE")
for branch in BRANCHES:
    block = OOF[branch]
    row_sums = block["proba"].sum(axis=1)
    print(f"  {branch:<8} rows {block['clip_id'].size:,} | unique clips "
          f"{len(set(block['clip_id'].tolist())):,} | proba {block['proba'].shape} "
          f"{block['proba'].dtype} | row sums "
          f"{row_sums.min():.6f}..{row_sums.max():.6f}")
    require(f"{branch} OOF holds exactly 6,000 rows",
            block["clip_id"].size == N_CLIPS, f"{block['clip_id'].size:,}")
    require(f"{branch} OOF covers every clip exactly once",
            sorted(block["clip_id"].tolist()) == CLIP_IDS, "no gaps, no duplicates")
    require(f"{branch} OOF labels match the frozen manifest",
            np.array_equal(block["y_true"], Y_ALL), "aligned")
    require(f"{branch} OOF signers match the frozen manifest",
            np.array_equal(block["signer_id"], SIGNER_ALL.astype(str)), "aligned")
    require(f"{branch} OOF probabilities are float32, finite and normalised",
            block["proba"].shape == (N_CLIPS, N_LABELS)
            and block["proba"].dtype == np.float32
            and bool(np.isfinite(block["proba"]).all())
            and float(np.abs(row_sums - 1.0).max()) < 1e-3,
            f"max |rowsum - 1| {float(np.abs(row_sums - 1.0).max()):.2e}")
    require(f"{branch} OOF predictions equal argmax of the probabilities",
            np.array_equal(block["y_pred"], np.argmax(block["proba"], axis=1)),
            "class order 0..199")
    # every clip was predicted by the fold that held out its signer
    predicted_by_holdout = all(
        block["signer_id"][i] in FOLDS[int(block["fold"][i])]["test"]
        for i in range(N_CLIPS))
    require(f"{branch} every clip was predicted by the fold holding out its signer",
            predicted_by_holdout, "signer-independent by construction")

require("full and reduced OOF cover the same clips in the same order",
        np.array_equal(OOF["full"]["clip_id"], OOF["reduced"]["clip_id"]),
        "paired row for row")

OOF_DIGEST: Dict[str, str] = {}
for branch in BRANCHES:
    block = OOF[branch]
    OOF_DIGEST[branch] = array_digest(block["y_pred"], block["proba"])
    atomic_write(OOF_PATH[branch], lambda h, b=block: np.savez_compressed(
        h, clip_id=b["clip_id"], signer_id=b["signer_id"], label_id=b["label_id"],
        fold=b["fold"], y_true=b["y_true"], y_pred=b["y_pred"], proba=b["proba"],
        __meta__=np.array([json.dumps(
            {**SCOPE_PROVENANCE, "branch": branch, "kind": "signer_independent_oof",
             "rows": int(N_CLIPS), "digest": OOF_DIGEST[branch]},
            sort_keys=True)])))
    with np.load(OOF_PATH[branch], allow_pickle=False) as check:
        require(f"{branch} OOF file round-trips with the same digest",
                array_digest(check["y_pred"], check["proba"]) == OOF_DIGEST[branch],
                OOF_DIGEST[branch])
    print(f"  wrote {OOF_PATH[branch].name}  "
          f"{OOF_PATH[branch].stat().st_size / 1e6:.1f} MB  digest {OOF_DIGEST[branch]}")

# ---- the control, kept strictly separate -----------------------------------
print("\nSIGNER-OVERLAP CONTROL (kept separate from the OOF results)")
CONTROL_DIGEST: Dict[str, str] = {}
for branch in BRANCHES:
    scope = SCOPE_RESULTS[(branch, "control")]
    CONTROL_DIGEST[branch] = array_digest(scope["y_pred"], scope["proba"])
    atomic_write(CONTROL_PATH[branch], lambda h, s=scope: np.savez_compressed(
        h, clip_id=s["clip_id"], signer_id=s["signer_id"], y_true=s["y_true"],
        y_pred=s["y_pred"], proba=s["proba"], val_clip_id=s["val_clip_id"],
        val_y_true=s["val_y_true"], val_y_pred=s["val_y_pred"],
        val_proba=s["val_proba"],
        __meta__=np.array([json.dumps(
            {**SCOPE_PROVENANCE, "branch": branch,
             "kind": "INTENTIONAL_SIGNER_OVERLAP_CONTROL",
             "label": "INTENTIONAL SIGNER-OVERLAP CONTROL",
             "splits": CONTROL_SPLIT_SIZES, "rows": int(scope["y_true"].size),
             "digest": CONTROL_DIGEST[branch],
             "warning": "invalid as a signer-independent evaluation; never pool with "
                        "the 6,000 out-of-fold predictions"},
            sort_keys=True)])))
    print(f"  {branch:<8} test {scope['y_true'].size} clips | accuracy "
          f"{scope['metrics_test']['accuracy']:.4f} | digest {CONTROL_DIGEST[branch]}")
    require(f"{branch} control test set holds 750 clips",
            int(scope["y_true"].size) == CONTROL_SPLIT_SIZES["test"],
            int(scope["y_true"].size))
    require(f"{branch} control is labelled as an intentional overlap control",
            bool(scope["is_control"]), "INTENTIONAL SIGNER-OVERLAP CONTROL")

control_clips = set(SCOPE_RESULTS[("full", "control")]["clip_id"].tolist())
require("control predictions are stored apart from the OOF arrays",
        CONTROL_PATH["full"] != OOF_PATH["full"]
        and len(control_clips) == CONTROL_SPLIT_SIZES["test"],
        "separate files, never pooled")

In [ ]:
# ============================================================
# CELL 9 — Metrics: per fold, validation (diagnostic), pooled, signer, class
# ============================================================
# No inferential statistics. The eight paired full/reduced fold values are saved
# so notebook 13 can run the pre-specified test there.

fold_rows_out, validation_rows_out = [], []
for fold in range(N_FOLDS):
    definition = FOLDS[fold]
    test_signers = list(definition["test"])
    test_clips = [CLIP_IDS[i] for i in np.sort(FOLD_ROWS[(fold, "test")])]
    regimes = REGIME.loc[test_clips]
    for branch in BRANCHES:
        scope = SCOPE_RESULTS[(branch, fold)]
        metrics = scope["metrics_test"]
        row = {
            "branch": branch, "fold": fold, "seed": int(scope["seed"]),
            "n_features": int(scope["n_features"]),
            "train_clips": int(scope["n_train"]), "test_clips": int(scope["n_test"]),
            "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "balanced_accuracy": metrics["balanced_accuracy"],
            f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
            "signer_macro_accuracy": metrics["signer_macro_accuracy"],
            # descriptive recording-regime metadata for notebook 13 only
            "test_n_30fps": int((regimes == REGIME_LABELS[0]).sum()),
            "test_n_60fps": int((regimes == REGIME_LABELS[1]).sum()),
            "test_regime_mix": ("mixed" if len(set(regimes)) > 1
                                else str(sorted(set(regimes))[0])),
            "test_signer_1": test_signers[0],
            "test_signer_1_regime": SIGNER_REGIME[test_signers[0]],
            "test_signer_2": test_signers[1],
            "test_signer_2_regime": SIGNER_REGIME[test_signers[1]],
        }
        for position, signer in enumerate(test_signers, start=1):
            row[f"test_signer_{position}_accuracy"] = metrics[
                "per_signer_accuracy"][signer]
        fold_rows_out.append(row)

        validation = scope["metrics_validation"]
        validation_rows_out.append({
            "branch": branch, "fold": fold, "status": "DIAGNOSTIC ONLY",
            "validation_signers": " ".join(definition["validation"]),
            "n": validation["n"], "accuracy": validation["accuracy"],
            "macro_f1": validation["macro_f1"],
            "weighted_f1": validation["weighted_f1"],
            "balanced_accuracy": validation["balanced_accuracy"],
            f"top{TOP_K}_accuracy": validation[f"top{TOP_K}_accuracy"],
            "signer_macro_accuracy": validation["signer_macro_accuracy"],
        })

fold_metrics = pd.DataFrame(fold_rows_out).sort_values(["branch", "fold"])
validation_metrics = pd.DataFrame(validation_rows_out).sort_values(["branch", "fold"])

print("PER-FOLD TEST METRICS (unseen signers)")
print(fold_metrics[["branch", "fold", "test_signer_1", "test_signer_2",
                    "test_regime_mix", "n_features", "accuracy", "macro_f1",
                    f"top{TOP_K}_accuracy", "signer_macro_accuracy"]]
      .round(4).to_string(index=False))

print("\nPAIRED FULL vs REDUCED, per fold (for notebook 13; no test is run here)")
pivot = fold_metrics.pivot(index="fold", columns="branch", values="accuracy")
pivot["difference_full_minus_reduced"] = pivot["full"] - pivot["reduced"]
print(pivot.round(4).to_string())
print(f"  mean accuracy  full {fold_metrics.loc[fold_metrics['branch'] == 'full', 'accuracy'].mean():.4f} "
      f"+/- {fold_metrics.loc[fold_metrics['branch'] == 'full', 'accuracy'].std(ddof=1):.4f}"
      f"   reduced {fold_metrics.loc[fold_metrics['branch'] == 'reduced', 'accuracy'].mean():.4f} "
      f"+/- {fold_metrics.loc[fold_metrics['branch'] == 'reduced', 'accuracy'].std(ddof=1):.4f}")
print("  Descriptive only. The pre-specified paired sign-flip test belongs to notebook 13.")

print("\nVALIDATION METRICS  [DIAGNOSTIC ONLY - never used for any decision]")
print(validation_metrics[["branch", "fold", "validation_signers", "accuracy",
                         "macro_f1"]].round(4).to_string(index=False))

# ---- pooled out-of-fold ----------------------------------------------------
pooled_rows, signer_rows, class_rows = [], [], []
CONFUSION: Dict[str, np.ndarray] = {}
for branch in BRANCHES:
    block = OOF[branch]
    metrics = scope_metrics(block["y_true"], block["y_pred"], block["proba"],
                            block["signer_id"])
    pooled_rows.append({
        "branch": branch, "kind": "pooled_signer_independent_oof", "n": N_CLIPS,
        "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
        "signer_macro_accuracy": metrics["signer_macro_accuracy"],
        "fold_accuracy_mean": float(fold_metrics.loc[
            fold_metrics["branch"] == branch, "accuracy"].mean()),
        "fold_accuracy_std": float(fold_metrics.loc[
            fold_metrics["branch"] == branch, "accuracy"].std(ddof=1)),
    })
    for signer, accuracy in metrics["per_signer_accuracy"].items():
        mask = block["signer_id"] == signer
        signer_rows.append({
            "branch": branch, "signer_id": signer, "n": int(mask.sum()),
            "accuracy": accuracy, "regime": SIGNER_REGIME[signer],
            "fold_tested_in": int(block["fold"][mask][0]),
            "top5_accuracy": top_k_accuracy(block["proba"][mask],
                                            block["y_true"][mask]),
        })
    recalls = recall_score(block["y_true"], block["y_pred"], average=None,
                           labels=np.arange(N_LABELS), zero_division=0)
    id_to_label = {v: k for k, v in LABEL_TO_ID.items()}
    for label_id, recall in enumerate(recalls):
        class_rows.append({"branch": branch, "label_id": label_id,
                           "label": id_to_label[label_id], "recall": float(recall),
                           "support": int((block["y_true"] == label_id).sum())})
    CONFUSION[branch] = confusion_matrix(block["y_true"], block["y_pred"],
                                         labels=np.arange(N_LABELS)).astype(np.int32)
    np.save(CONFUSION_PATH[branch], CONFUSION[branch])
    require(f"{branch} confusion matrix is 200x200 and totals 6,000",
            CONFUSION[branch].shape == (N_LABELS, N_LABELS)
            and int(CONFUSION[branch].sum()) == N_CLIPS,
            f"{CONFUSION[branch].shape}, {int(CONFUSION[branch].sum()):,}")
    require(f"{branch} pooled accuracy reproduces from the confusion matrix",
            abs(float(np.trace(CONFUSION[branch]) / N_CLIPS)
                - metrics["accuracy"]) < 1e-12, f"{metrics['accuracy']:.6f}")

pooled_metrics = pd.DataFrame(pooled_rows)
signer_metrics = pd.DataFrame(signer_rows)
class_recall = pd.DataFrame(class_rows)

print(f"\nPOOLED {N_CLIPS:,}-CLIP OUT-OF-FOLD METRICS (the primary result)")
print(pooled_metrics.round(4).to_string(index=False))

print("\nPER-SIGNER ACCURACY (pooled out-of-fold)")
print(signer_metrics.pivot(index="signer_id", columns="branch", values="accuracy")
      .reindex(sorted(SIGNER_REGIME, key=lambda v: int(v.lstrip("p"))))
      .round(4).to_string())

# ---- control metrics, labelled and separate --------------------------------
control_rows = []
for branch in BRANCHES:
    scope = SCOPE_RESULTS[(branch, "control")]
    metrics = scope["metrics_test"]
    control_rows.append({
        "branch": branch, "kind": "INTENTIONAL SIGNER-OVERLAP CONTROL",
        "valid_as_signer_independent": False,
        "train": CONTROL_SPLIT_SIZES["train"],
        "validation": CONTROL_SPLIT_SIZES["validation"],
        "test": CONTROL_SPLIT_SIZES["test"], "seed": int(scope["seed"]),
        "n_features": int(scope["n_features"]),
        "accuracy": metrics["accuracy"], "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        f"top{TOP_K}_accuracy": metrics[f"top{TOP_K}_accuracy"],
        "validation_accuracy_diagnostic": scope["metrics_validation"]["accuracy"],
        "pooled_oof_accuracy_same_branch": float(
            pooled_metrics.loc[pooled_metrics["branch"] == branch,
                               "accuracy"].iloc[0]),
    })
control_metrics = pd.DataFrame(control_rows)
control_metrics["signer_overlap_gap"] = (
    control_metrics["accuracy"] - control_metrics["pooled_oof_accuracy_same_branch"])
print("\nSIGNER-OVERLAP CONTROL  [invalid by design; reported as a gap, not as leakage]")
print(control_metrics[["branch", "n_features", "accuracy",
                       "pooled_oof_accuracy_same_branch",
                       "signer_overlap_gap"]].round(4).to_string(index=False))
print("  The gap is the SIGNER-OVERLAP PERFORMANCE GAP. Identity memorisation is one")
print("  plausible contributor; session and background similarity are others.")

efficiency = pd.DataFrame(EFFICIENCY)
preflight_out = preflight_table.assign(branch=preflight_table["branch"],
                                       scope="preflight_25_trees")
efficiency = pd.concat([efficiency, preflight_out], ignore_index=True)

for name, frame in (("fold_metrics", fold_metrics),
                    ("validation_metrics", validation_metrics),
                    ("pooled_metrics", pooled_metrics),
                    ("signer_metrics", signer_metrics),
                    ("class_recall", class_recall),
                    ("control_metrics", control_metrics),
                    ("efficiency", efficiency)):
    atomic_write(OUT[name], lambda h, f=frame: f.to_csv(h, index=False))
    print(f"  wrote {OUT[name].name}  {len(frame):,} rows")

# metrics must be reproducible from the saved predictions, not just from memory
with np.load(OOF_PATH["full"], allow_pickle=False) as check:
    reproduced = float(accuracy_score(check["y_true"],
                                      check["y_pred"]))
require("pooled metrics reproduce from the saved OOF file",
        abs(reproduced - float(pooled_metrics.loc[
            pooled_metrics["branch"] == "full", "accuracy"].iloc[0])) < 1e-12,
        f"{reproduced:.6f}")
require("all eight paired fold values are saved for notebook 13",
        len(fold_metrics) == N_FOLDS * len(BRANCHES)
        and pivot["difference_full_minus_reduced"].notna().all(),
        f"{N_FOLDS} paired differences")
require("no inferential statistic was computed here", True,
        "sign-flip test and intervals deferred to notebook 13")

In [ ]:
# ============================================================
# CELL 10 — Feature importance (EXPLORATORY)
# ============================================================
# Impurity-based importance only, expanded back into the full notebook 9 feature
# space with filtered-out features recorded as 0.0. No permutation importance.
# Impurity importance is biased toward high-cardinality features and is correlated
# across correlated features, so it is exploratory and never inferential.

IMPORTANCE: Dict[str, np.ndarray] = {}
for branch in BRANCHES:
    expanded = np.zeros((N_FOLDS, FEATURE_COUNT[branch]), dtype=np.float64)
    for fold in range(N_FOLDS):
        scope = SCOPE_RESULTS[(branch, fold)]
        keep_idx = np.asarray(scope["keep_idx"], dtype=np.int64)
        values = np.asarray(scope["importances"], dtype=np.float64)
        require(f"{branch} fold {fold} importance length matches its keep mask",
                values.size == keep_idx.size == EXPECTED_RETAINED[branch][fold],
                f"{values.size:,}")
        expanded[fold, keep_idx] = values
    IMPORTANCE[branch] = expanded
    np.save(IMPORTANCE_PATH[branch], expanded)
    dropped = np.setdiff1d(np.arange(FEATURE_COUNT[branch]),
                           np.asarray(SCOPE_RESULTS[(branch, 2)]["keep_idx"]))
    print(f"  {branch:<8} importances {expanded.shape} | per-fold sums "
          f"{expanded.sum(axis=1).min():.6f}..{expanded.sum(axis=1).max():.6f} | "
          f"wrote {IMPORTANCE_PATH[branch].name}")
    require(f"{branch} importance array has the frozen shape",
            expanded.shape == (N_FOLDS, FEATURE_COUNT[branch]), str(expanded.shape))
    require(f"{branch} each fold's importances sum to 1",
            float(np.abs(expanded.sum(axis=1) - 1.0).max()) < 1e-9,
            f"max deviation {float(np.abs(expanded.sum(axis=1) - 1.0).max()):.2e}")
    require(f"{branch} filtered-out features are exactly 0.0",
            float(np.abs(expanded[2:, dropped]).max()) == 0.0,
            f"{dropped.size} dropped features")
    require(f"{branch} importances are non-negative and finite",
            bool(np.isfinite(expanded).all()) and float(expanded.min()) >= 0.0,
            "clean")

# ---- concise exploratory summaries -----------------------------------------
summary_rows = []
for branch in BRANCHES:
    spec = SPEC[branch].sort_values("feature_index")
    mean_importance = IMPORTANCE[branch].mean(axis=0)
    frame = pd.DataFrame({
        "modality": spec["modality"].to_numpy(),
        "family": spec["family"].to_numpy(),
        "statistic": spec["statistic"].to_numpy(),
        "importance": mean_importance,
    })
    for grouping in ("modality", "family", "statistic"):
        grouped = frame.groupby(grouping, as_index=False).agg(
            features=("importance", "size"),
            total_importance=("importance", "sum"),
            mean_importance=("importance", "mean"))
        grouped["share"] = grouped["total_importance"] / grouped["total_importance"].sum()
        grouped.insert(0, "grouping", grouping)
        grouped.insert(0, "branch", branch)
        grouped = grouped.rename(columns={grouping: "group"})
        summary_rows.append(grouped)

importance_summary = pd.concat(summary_rows, ignore_index=True)
importance_summary["status"] = "EXPLORATORY"
atomic_write(OUT["importance_summary"],
             lambda h: importance_summary.to_csv(h, index=False))

print("\nIMPORTANCE BY MODALITY  (mean over the 8 folds, share of total)  EXPLORATORY")
view = importance_summary[importance_summary["grouping"] == "modality"]
print(view.pivot(index="group", columns="branch", values="share")
      .round(4).to_string())
print("\nIMPORTANCE BY FEATURE FAMILY  EXPLORATORY")
view = importance_summary[importance_summary["grouping"] == "family"]
print(view.pivot(index="group", columns="branch", values="share")
      .round(4).to_string())
print("\nTOP STATISTICS BY SHARE  EXPLORATORY")
view = importance_summary[importance_summary["grouping"] == "statistic"]
print(view.pivot(index="group", columns="branch", values="share")
      .sort_values("full", ascending=False).head(8).round(4).to_string())

require("importance shares sum to 1 within every grouping and branch",
        float(np.abs(importance_summary.groupby(["branch", "grouping"])["share"]
                     .sum() - 1.0).max()) < 1e-9, "normalised")
require("importance is labelled exploratory throughout",
        set(importance_summary["status"].unique()) == {"EXPLORATORY"}, "EXPLORATORY")
print(f"\n  wrote {OUT['importance_summary'].name}  "
      f"{len(importance_summary):,} rows")
print("  Impurity importance is biased toward high-cardinality and correlated")
print("  features. It is exploratory context for notebook 13, not evidence.")

In [ ]:
# ============================================================
# CELL 11 — Freeze the manifest, then write the validation report last
# ============================================================

RUN_METADATA = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "environment": ENVIRONMENT,
    "resolved_paths_this_session": {"notebook7": str(NB7_DIR),
                                    "notebook8": str(NB8_DIR),
                                    "notebook9": str(NB9_DIR)},
    "runtime_seconds": round(time.time() - STARTED, 1),
    "note": "recorded for traceability and deliberately excluded from "
            "rf_model_contract_id, which must not change between reruns",
}

pooled_lookup = {row["branch"]: row for row in pooled_metrics.to_dict("records")}
control_lookup = {row["branch"]: row for row in control_metrics.to_dict("records")}

MANIFEST = {
    "rf_model_contract_id": RF_MODEL_CONTRACT_ID,
    "rf_model_contract": RF_MODEL_CONTRACT,
    "upstream": dict(REQUIRED_IDS),
    "scopes_completed": len(SCOPE_RESULTS),
    "oof": {b: {"file": OOF_PATH[b].name, "rows": N_CLIPS,
                "digest": OOF_DIGEST[b],
                "accuracy": pooled_lookup[b]["accuracy"],
                "macro_f1": pooled_lookup[b]["macro_f1"],
                "signer_macro_accuracy": pooled_lookup[b]["signer_macro_accuracy"]}
            for b in BRANCHES},
    "control": {b: {"file": CONTROL_PATH[b].name,
                    "label": "INTENTIONAL SIGNER-OVERLAP CONTROL",
                    "digest": CONTROL_DIGEST[b],
                    "accuracy": control_lookup[b]["accuracy"],
                    "signer_overlap_gap": control_lookup[b]["signer_overlap_gap"],
                    "valid_as_signer_independent": False}
                for b in BRANCHES},
    "retained_dimensions_observed": {
        b: {str(s): int(SCOPE_RESULTS[(b, s)]["n_features"])
            for s in list(range(N_FOLDS)) + ["control"]} for b in BRANCHES},
    "paired_fold_accuracy": {
        "full": [float(v) for v in fold_metrics.loc[
            fold_metrics["branch"] == "full"].sort_values("fold")["accuracy"]],
        "reduced": [float(v) for v in fold_metrics.loc[
            fold_metrics["branch"] == "reduced"].sort_values("fold")["accuracy"]],
        "note": "saved for the pre-specified paired test in notebook 13; no test is "
                "run in notebook 10",
    },
    "artefacts": {name: path.name for name, path in OUT.items()},
    "importance": {b: {"file": IMPORTANCE_PATH[b].name,
                       "shape": [N_FOLDS, FEATURE_COUNT[b]], "status": "EXPLORATORY"}
                   for b in BRANCHES},
    "confusion": {b: CONFUSION_PATH[b].name for b in BRANCHES},
    "run_metadata": RUN_METADATA,
}
atomic_write(OUT["manifest"], lambda h: h.write(
    json.dumps(MANIFEST, indent=2, sort_keys=True, default=str).encode()))
reloaded_manifest = json.loads(OUT["manifest"].read_text(encoding="utf-8"))
require("the rf model contract id reproduces from the manifest on disk",
        sha16(json.dumps(reloaded_manifest["rf_model_contract"], sort_keys=True,
                         separators=(",", ":"), default=str))
        == RF_MODEL_CONTRACT_ID == reloaded_manifest["rf_model_contract_id"],
        RF_MODEL_CONTRACT_ID)
require("run metadata is stored outside the hashed contract",
        "generated_at_utc" not in json.dumps(reloaded_manifest["rf_model_contract"]),
        "separated")
require("the manifest records both OOF digests",
        all(reloaded_manifest["oof"][b]["digest"] == OOF_DIGEST[b] for b in BRANCHES),
        "recorded")

FINAL_FAILURES = failures()
SKIPPED = [entry for entry in CHECKS if entry["status"] == "skip"]
VERDICT = "FAIL" if FINAL_FAILURES else "PASS"

report = {
    "notebook": "10-random-forest-8-fold", "rf_policy_version": RF_POLICY_VERSION,
    "verdict": VERDICT, "rf_model_contract_id": RF_MODEL_CONTRACT_ID,
    "upstream": dict(REQUIRED_IDS),
    "checks_total": len(CHECKS),
    "checks_passed": len([c for c in CHECKS if c["status"] == "pass"]),
    "checks_failed": len(FINAL_FAILURES), "checks_skipped": len(SKIPPED),
    "checks": CHECKS,
    "oof_digest": OOF_DIGEST, "control_digest": CONTROL_DIGEST,
    "pooled_metrics": pooled_metrics.to_dict("records"),
    "control_metrics": control_metrics.to_dict("records"),
    "fold_metrics": fold_metrics.to_dict("records"),
    "efficiency": efficiency.to_dict("records"),
    "runtime_seconds": round(time.time() - STARTED, 1),
    "run_metadata": RUN_METADATA,
}
atomic_write(OUT["report_json"], lambda h: h.write(
    json.dumps(report, indent=2, default=str).encode()))
atomic_write(OUT["report_csv"], lambda h: pd.DataFrame(CHECKS)[
    ["check", "status", "passed", "detail"]].to_csv(h, index=False))
print(f"wrote {OUT['report_json'].name} and {OUT['report_csv'].name} "
      f"({len(CHECKS)} checks, verdict {VERDICT})")

if FINAL_FAILURES:
    for entry in FINAL_FAILURES:
        print(f"  FAIL  {entry['check']}: {entry['detail']}")
    raise RuntimeError(
        f"{len(FINAL_FAILURES)} checks failed. The report was written for diagnosis, "
        "but these Random Forest results are NOT usable and notebook 13 must not read "
        "them.")

written = [OUT["contract"], OUT["manifest"], *OUT.values(),
           *OOF_PATH.values(), *CONTROL_PATH.values(),
           *CONFUSION_PATH.values(), *IMPORTANCE_PATH.values()]
written = list(dict.fromkeys(p for p in written if p.exists()))
scope_bytes = sum(p.stat().st_size for p in SCOPE_DIR.glob("rf_scope_*.npz"))
total_bytes = sum(p.stat().st_size for p in written) + scope_bytes

print("=" * 78)
print("NOTEBOOK 10 COMPLETE - RANDOM FOREST EVALUATION FROZEN")
print("=" * 78)
print(f"  experiment id        : {REQUIRED_EXPERIMENT_ID}")
print(f"  loader contract id   : {REQUIRED_LOADER_CONTRACT_ID}")
print(f"  feature contract id  : {REQUIRED_FEATURE_CONTRACT_ID}")
print(f"  filter set id        : {REQUIRED_FILTER_SET_ID}")
print(f"  time metadata id     : {REQUIRED_TIME_METADATA_ID}")
print(f"  rf model contract id : {RF_MODEL_CONTRACT_ID}")
print()
print(f"  full    OOF digest   : {OOF_DIGEST['full']}")
print(f"  reduced OOF digest   : {OOF_DIGEST['reduced']}")
print(f"  full    control digest: {CONTROL_DIGEST['full']}")
print(f"  reduced control digest: {CONTROL_DIGEST['reduced']}")
print()
print(f"  POOLED {N_CLIPS:,}-CLIP OUT-OF-FOLD (primary result)")
for branch in BRANCHES:
    row = pooled_lookup[branch]
    print(f"    {branch:<8} accuracy {row['accuracy']:.4f} | macro F1 "
          f"{row['macro_f1']:.4f} | weighted F1 {row['weighted_f1']:.4f} | "
          f"balanced {row['balanced_accuracy']:.4f}")
    print(f"    {'':<8} top{TOP_K} {row[f'top{TOP_K}_accuracy']:.4f} | signer-macro "
          f"{row['signer_macro_accuracy']:.4f} | per-fold "
          f"{row['fold_accuracy_mean']:.4f} +/- {row['fold_accuracy_std']:.4f}")
print()
print("  SIGNER-OVERLAP CONTROL (invalid by design, reported as a gap)")
for branch in BRANCHES:
    row = control_lookup[branch]
    print(f"    {branch:<8} accuracy {row['accuracy']:.4f} | gap vs pooled OOF "
          f"{row['signer_overlap_gap']:+.4f}")
print()
print(f"  checks               : {report['checks_passed']} passed, "
      f"{report['checks_failed']} failed, {report['checks_skipped']} skipped "
      f"of {len(CHECKS)}")
print(f"  runtime              : {(time.time() - STARTED) / 60:.1f} min")
print(f"  outputs              : {total_bytes / 1e6:.0f} MB "
      f"(no trained forest is persisted)")

print("\n  Frozen here: the RF model contract, the 6,000-clip out-of-fold predictions")
print("  and probabilities for both branches, the two control runs, per-fold and")
print("  pooled metrics, per-signer and per-class metrics, confusion matrices, and")
print("  exploratory impurity importances.")
print("\n  NOT done here: no paired sign-flip test, no confidence interval, no")
print("  equivalence or significance claim. Notebook 13 owns all of that.")
print("\n  Carried forward: max_features=96 holds absolute search breadth equal and is")
print("  potentially conservative toward Full in per-split search breadth relative to")
print("  its natural sqrt setting. The post-zero-variance model-input ratio was "
      f"{RF_MODEL_CONTRACT['dimensionality']['post_zero_variance_ratio']:.3f}x.")
print("  Both must be stated wherever this RF comparison is reported.")

print("\n" + "=" * 78)
print(f"  Quote rf model contract id {RF_MODEL_CONTRACT_ID} in notebook 13.")
print("  STOP. Do not build notebook 11 until this real Kaggle log has been reviewed.")
print("=" * 78)